# Climate DT and CMIP6

In this exercise we will download CMIP6 data and compare it to Climate DT data. CMIP6 is the [Climate Model Intercomparison Project Phase 6](https://wcrp-cmip.org/cmip-phases/cmip6/) which includes many different model experiments, mostly with resoltions coarser than 100km. These experiments can be a interesting to use as a reference point for current lower resolution models. Comparing to those one can see where we can benefit from high-resolution data but also where resolution increases do not solve all issues in climate models. In this exercise we will look at the Uusimaa area as the example of a metropolitan area where climate adaptation on the local scale can profit from high-resolution data.

**Steps in this exercise:**  
- Download CMIP6 data selection
- Download a data selection of the Climate DT SSP3-7.0 simulations
- Regrid standard resolution healpix data to CMIP6 grid
- Compare variable X and Y of Climate DT to the multi-model mean of CMIP6
- 

## Downloading CMIP6 Data for Southern Finland

This section demonstrates how to access and download CMIP6 data for southern Finland using the [Pangeo CMIP6 cloud catalog](https://pangeo-data.github.io/pangeo-cmip6-cloud/). The data is stored in Zarr format on Google Cloud Storage, so no ESGF account is needed.

**Southern Finland bounding box used in this example:**
- Latitude: 59.5°N – 61.5°N
- Longitude: 20.0°E – 32.0°E

This covers the Uusimaa region and surrounding areas including Helsinki, Turku, and Tampere.

### Required Packages

Install the following packages before running the cells below:

```bash
pip install intake intake-esm xarray zarr gcsfs pandas matplotlib cartopy
```

| Package | Purpose |
|---|---|
| `intake` | Data catalogue framework |
| `intake-esm` | ESM (Earth System Model) catalogue extension for CMIP6 |
| `xarray` | N-dimensional labelled arrays and datasets |
| `zarr` | Chunked, compressed array storage (format used by Pangeo) |
| `gcsfs` | Google Cloud Storage filesystem access |
| `pandas` | Tabular data handling for catalogue queries |
| `matplotlib` | Plotting |
| `cartopy` | Geospatial projections and map plotting |

In [ ]:
import intake
import intake_esm
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

### Step 1: Open the Pangeo CMIP6 Catalogue

The Pangeo catalogue indexes thousands of CMIP6 datasets available on Google Cloud Storage. We query it to find datasets matching our experiment, variable, and model of interest.

In [ ]:
# Open the Pangeo CMIP6 catalogue (hosted on Google Cloud Storage)
catalog_url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(catalog_url)

print(col)
print(f"\nTotal datasets available: {len(col.df)}")
#https://intake-esm.readthedocs.io/en/stable/tutorials/loading-cmip6-data.html

### Step 2: Search the Catalogue

Filter the catalogue by:
- **experiment_id**: `ssp370` — the SSP3-7.0 matching the Climate DT scenario used in this workshop
- **variable_id**: `tas` — near-surface air temperature (2m temperature)
- **table_id**: `Amon` — monthly atmospheric data
- **member_id**: `r1i1p1f1` — the first ensemble member (consistent across models)

In [ ]:
query = dict(
    experiment_id="ssp370",   # SSP2-4.5 scenario
    variable_id="tas",         # Near-surface air temperature
    table_id="Amon",           # Monthly atmosphere
    member_id="r1i1p1f1",      # First ensemble member
)

cat = col.search(**query)
print(f"Datasets found: {len(cat.df)}")
print("\nAvailable models:")
print(cat.df["source_id"].unique())

### Step 3: Load Datasets and Subset to Southern Finland

We load the datasets into a dictionary of `xarray.Dataset` objects, then immediately subset to the southern Finland bounding box to keep memory usage low. Computations are lazy (Dask-backed) until `.load()` or `.compute()` is called.

In [ ]:
# Bounding box for southern Finland
LAT_MIN, LAT_MAX = 59.5, 61.5
LON_MIN, LON_MAX = 20.0, 32.0

def subset_southern_finland(ds):
    """Subset an xarray Dataset to the southern Finland bounding box."""
    # Longitude may be stored as 0–360 or -180–180; normalise to 0–360
    if ds.lon.min() < 0:
        ds = ds.assign_coords(lon=(ds.lon % 360)).sortby("lon")
    return ds.sel(
        lat=slice(LAT_MIN, LAT_MAX),
        lon=slice(LON_MIN, LON_MAX),
    )

# Load datasets (lazy — actual data is not downloaded yet)
dset_dict = cat.to_dataset_dict(
    xarray_open_kwargs={"consolidated": True, "use_cftime": True}
)

# Subset each dataset to southern Finland
dset_finland = {}
for key, ds in dset_dict.items():
    try:
        dset_finland[key] = subset_southern_finland(ds)
        print(f"  {key}: {dset_finland[key].dims}")
    except Exception as e:
        print(f"  {key}: skipped ({e})")

print(f"\nLoaded {len(dset_finland)} datasets for southern Finland.")

### Step 4: Compute Spatial Mean and Save to NetCDF

Compute the area-weighted spatial mean over southern Finland for each model and optionally save the result to a local NetCDF file for later use.

In [ ]:
import numpy as np

def area_weighted_mean(ds, var="tas"):
    """Compute cosine-latitude-weighted spatial mean of a variable."""
    weights = np.cos(np.deg2rad(ds.lat))
    weights.name = "weights"
    return ds[var].weighted(weights).mean(dim=["lat", "lon"])

# Compute spatial means — this triggers the actual data download
timeseries = {}
for key, ds in dset_finland.items():
    print(f"Computing mean for {key} ...")
    timeseries[key] = area_weighted_mean(ds).compute()

print("Done.")

### Step 5: Save to NetCDF

Save the subsetted spatial data (full lat/lon grid, not just the mean) to a local NetCDF file for use in the rest of this exercise.

In [ ]:
# Save each model's southern Finland subset to a separate NetCDF file
for key, ds in dset_finland.items():
    # key format: "activity_id.institution_id.source_id.experiment_id...."
    source_id = ds.attrs.get("source_id", key.split(".")[2])
    out_path = f"cmip6_tas_southern_finland_{source_id}.nc"
    ds.compute().to_netcdf(out_path)
    print(f"Saved {out_path}")

### Step 6: Quick Verification Plot

Plot the spatial mean temperature time series for all models to verify the download looks sensible before proceeding with the comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for key, ts in timeseries.items():
    source_id = key.split(".")[2]
    # Convert from Kelvin to Celsius
    ax.plot(ts.time.values, ts.values - 273.15, label=source_id, alpha=0.7, linewidth=0.8)

ax.set_title("CMIP6 SSP2-4.5 — Near-surface Temperature, Southern Finland")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Year")
ax.legend(fontsize=7, ncol=3, loc="upper left")
plt.tight_layout()
plt.show()